# Pre

In [ ]:
import pandas as pd
import numpy as np

def load_data(X_path, Y_path=None):
    # -- Load the labelled data
    df = pd.read_csv(X_path, sep=r"\s+", engine="python")
    print(df.head())
    df.columns = df.columns.str.replace('"', '', regex=False) # Column names have double quotes
    meta = df[["Chromosome", "Start", "End", "Nclone"]]
    X = df.filter(regex=r"^Array\.") 
    X = X.T # Each Array.* from data represents one instance
    print(X.head())

    Y = None
    if Y_path is not None:
        Y = pd.read_csv(Y_path, sep='\t')
        Y.columns = ['Sample', 'Subgroup']
        print(Y.head())
 
    return X, Y

X, Y = load_data("data/Train_call.tsv", "data/Train_clinical.tsv")

X_validation = load_data("data/Validation_call.tsv")[0]

# Model pipeline

In [ ]:
import pickle
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import make_pipeline
import numpy as np

# def reduce_features(X, X_validation, new_size=100): # is this safe
#     from sklearn.feature_selection import SelectKBest, f_classif

#     print("Reducing features using ANOVA F-value for efficiency")
#     print(f"Original size > {X.shape}")

#     # Filter: Keep ONLY the x best features based on ANOVA F-value
#     k_best_selector = SelectKBest(f_classif, k=new_size)
#     X_top = k_best_selector.fit_transform(X, Y["Subgroup"])

#     X_validation_top = k_best_selector.transform(X_validation)

#     print(f"Reduced size > {X_top.shape}")

#     return X_top, X_validation_top


def run_classification(classifier, classifier_grid, selector, selector_grid, text="", inner_splits=10, outer_splits=3, repeats=10, reduced_size=250):
    print(f"\n\n ------ {text} ------")

    # global X, Y, X_validation
    # # reduce features for efficiency for computationally expensive selectors
    # if ("Forward" in text) or ("Mutual" in text):
    #     X, X_validation = reduce_features(X, X_validation, new_size=reduced_size)

    # -- steps of the model
    pipeline = make_pipeline(selector, classifier)

    # -- hyperparameters to test
    params_grid = {
        selector_grid[0]: selector_grid[1],
        classifier_grid[0]: classifier_grid[1]
    }

    all_outer_scores = []

    # Repeat the nested CV
    for i in range(repeats):
        
        # -- Inner loop: train the model and evaluate the performance
        inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=i)
        grid_search = GridSearchCV(
            estimator=pipeline,  # selector then classifier
            param_grid=params_grid,
            cv=inner_cv,
            scoring="accuracy",
            n_jobs=-1,
            return_train_score=True
        )

        # -- Outer loop: repeat the inner loop and report the average performance
        outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=i + 100)
        nested_scores = cross_val_score(
            grid_search,
            X, Y["Subgroup"],
            cv=outer_cv,
            n_jobs=-1
        )

        print(f"Run {i+1}/{repeats} - Outer scores: {nested_scores}")
        all_outer_scores.append(nested_scores)

    # Refit the model on the entire dataset to get the best hyperparameters and make predictions on the unseen data
    grid_search.fit(X, Y["Subgroup"])           # now grid_search is fitted on full data
    best_model = grid_search.best_estimator_
    final_predictions = best_model.predict(X_validation)

    stats = {
        "Run details": text,
        "Average Performance (V)": np.mean(np.concatenate(all_outer_scores)),  # scalar over all repeats
        "Outer scores per repeat": all_outer_scores,
        "Best parameters": grid_search.best_params_,
        "Final predictions on unseen data": final_predictions,
        "Final model": best_model
    }

    print(stats)
    return stats

# Classifiers

In [ ]:
        "Average Performance (V)": np.mean(np.concatenate(all_outer_scores)),  # scalar over all repeats
        "Outer scores per repeat": all_outer_scores,
        "Best parameters": grid_search.best_params_,
        "Final predictions on unseen data": final_predictions,
        "Final model": best_model
    }

    print(stats)
    return stats

# Selectors

In [ ]:
# Forward Selection
from sklearn.feature_selection import SequentialFeatureSelector
selector_forward = SequentialFeatureSelector(estimator=classifier_knn, direction="forward", cv=2, n_features_to_select=10, scoring="accuracy")
selector_grid_forward = ('sequentialfeatureselector__n_features_to_select', [2, 5, 10, 25])

# Variance Threshold
from sklearn.feature_selection import VarianceThreshold
selector_variance = VarianceThreshold(threshold=0.01)
selector_grid_variance = ('variancethreshold__threshold', [0.001, 0.01, 0.05, 0.1])

# ANOVA
from sklearn.feature_selection import SelectKBest, f_classif
selector_anova = SelectKBest(score_func=f_classif, k=10)
selector_grid_anova = ('selectkbest__k', [2, 5, 10, 25])

# Mutual Information
from sklearn.feature_selection import SelectKBest, mutual_info_classif
selector_mi = SelectKBest(score_func=mutual_info_classif, k=10)
selector_grid_mi = ('selectkbest__k', [2, 5, 10, 25])

# PCA
from sklearn.decomposition import PCA
selector_pca = PCA(n_components=10)
selector_grid_pca = ('pca__n_components', [2, 5, 10, 25])

Selectors = [
(selector_variance, selector_grid_variance, "Variance Threshold"),
(selector_anova, selector_grid_anova, "ANOVA"),
(selector_pca, selector_grid_pca, "PCA"),
(selector_mi, selector_grid_mi, "Mutual Information"),

#(selector_forward, selector_grid_forward, "Forward Selection"),
 ]

# Code runs

In [ ]:
import pandas as pd
from pathlib import Path

inner_splits = 5
outer_splits = 3
repeats = 1
reduced_size = 250

def run_all_combinations(title="All Classifiers with All Selectors"):

    # save results
    results_dir = Path("results")
    results_dir.mkdir(parents=True, exist_ok=True)
    timestamp = pd.Timestamp.now().strftime("(%d_%H:%M)")
    path = results_dir / f"{title} {timestamp} in{inner_splits} out{outer_splits} rep{repeats}.tsv"

    for selector, selector_grid, selector_name in Selectors: # eaach combination
        for classifier, classifier_grid, classifier_name in Classifiers:
            stats = run_classification(
                classifier,
                classifier_grid,
                selector,
                selector_grid, 
                text=f"{classifier_name} with {selector_name}",
                inner_splits=inner_splits,
                outer_splits=outer_splits,
                repeats=repeats,
                reduced_size=reduced_size
            )


            data = {
                "classifier": classifier_name,
                "selector": selector_name,
                "avg_performance_v": stats.get("Average Performance (V)"),
                "outer_scores_per_repeat": str(stats.get("Outer scores per repeat")),
                "best_parameters": str(stats.get("Best parameters")),
                "final_predictions_unseen": str(stats.get("Final predictions on unseen data"))
            }

            # Save data
            df = pd.DataFrame([data])
            df.to_csv(path, sep="\t", index=False, mode='a', header=not path.exists())  # Append to file, write header only if file doesn't exist

            # Save the final model
            with open(results_dir / f"{title}_{classifier_name}_{selector_name}_model.pkl", "wb") as f:
                pickle.dump(stats.get("Final model"), f)

    return path

run_all_combinations(title="Test")